# ArmGPT - Build Your Own Armenian Language Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EdikSimonian/armenian-gpt/blob/main/notebooks/armgpt_colab.ipynb)

In this notebook, you will:
1. Download Armenian text data (from Wikipedia)
2. Build a vocabulary (mapping characters to numbers)
3. Train a GPT language model
4. Generate Armenian text!

**No prior experience needed** - just run each cell from top to bottom.

## Step 0: Setup

First, let's install the required packages and get the code.

In [ ]:
# Clone the ArmGPT repository (or pull latest changes if already cloned)
import os
os.chdir('/content')
if os.path.exists('armenian-gpt'):
    os.chdir('armenian-gpt')
    !git pull
else:
    !git clone https://github.com/EdikSimonian/armenian-gpt.git
    os.chdir('armenian-gpt')

# Install dependencies
!pip install -q torch numpy requests

In [ ]:
# Check if we have a GPU (free on Colab!)
import torch
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    total_mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"GPU memory: {total_mem / 1e9:.1f} GB")
    device = 'cuda'
else:
    print("No GPU found. Training will be slower but still works!")
    device = 'cpu'

## Step 1: Download Armenian Text Data

We'll download text from Armenian Wikipedia. This is a free collection of
articles written in Armenian, covering history, science, culture, and more.

**This step takes ~5 minutes** (downloading ~500 MB).

In [ ]:
# Download Armenian Wikipedia
!python data/download.py

## Step 2: Prepare the Data

Now we'll:
- Clean the text (remove HTML, keep only Armenian characters)
- Build a **vocabulary** (a list of every unique character)
- Convert text to numbers (because neural networks work with numbers!)
- Split into training data (90%) and validation data (10%)

In [ ]:
# Prepare the data with character-level tokenization
!python data/prepare.py --tokenizer char

### Let's explore the data!

In [ ]:
import json
import numpy as np

# Load the tokenizer to see our vocabulary
with open('data/tokenizer.json', 'r', encoding='utf-8') as f:
    tok_data = json.load(f)

vocab = tok_data['itos']
print(f"Vocabulary size: {len(vocab)} characters")
print(f"\nAll characters in our vocabulary:")
for i, ch in enumerate(vocab):
    display = repr(ch) if ch in (' ', '\n', '\t') else ch
    print(f"  {i:3d} -> {display}", end='    ')
    if (i + 1) % 5 == 0:
        print()

In [ ]:
# Look at the training data
train_data = np.memmap('data/train.bin', dtype=np.uint16, mode='r')
val_data = np.memmap('data/val.bin', dtype=np.uint16, mode='r')

print(f"Training tokens:   {len(train_data):,}")
print(f"Validation tokens: {len(val_data):,}")

# Decode a sample from the training data
sample_ids = train_data[:500].tolist()
sample_text = ''.join(vocab[i] for i in sample_ids if i < len(vocab))
print(f"\nSample from training data:\n")
print(sample_text)

In [ ]:
# Visualize character frequency
try:
    import matplotlib.pyplot as plt
    
    # Count how often each character appears
    counts = np.bincount(train_data, minlength=len(vocab))
    
    # Plot top 40 characters
    top_n = 40
    top_indices = np.argsort(counts)[-top_n:][::-1]
    top_chars = [repr(vocab[i]) if vocab[i] in (' ', '\n') else vocab[i] 
                 for i in top_indices if i < len(vocab)]
    top_counts = [counts[i] for i in top_indices if i < len(vocab)]
    
    plt.figure(figsize=(14, 5))
    plt.bar(range(len(top_chars)), top_counts)
    plt.xticks(range(len(top_chars)), top_chars, fontsize=10)
    plt.ylabel('Frequency')
    plt.title('Most Common Characters in Armenian Training Data')
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not available - skipping visualization")

## Step 3: Train the Model

Now for the exciting part! We'll create a GPT model and train it on Armenian text.

**What the model learns:** Given a sequence of characters, predict the next one.
Over thousands of training steps, it gets better and better at this task.

Watch the **loss** go down - that means the model is learning!
- Starting loss: ~4.5 (random guessing)
- Good loss: < 2.0 (the model has learned Armenian patterns)

In [ ]:
# Choose your training size:
# - 'tiny'   : ~1 min on CPU, basic results (good for testing)
# - 'small'  : ~30 min on GPU, decent results (recommended)
# - 'medium' : ~2 hours on GPU, best results

PRESET = 'small'  # <-- change this!

!python train.py --preset {PRESET}

### Visualize Training Progress

In [ ]:
try:
    import matplotlib.pyplot as plt
    
    # Load metrics
    with open('checkpoints/metrics.json', 'r') as f:
        metrics = json.load(f)
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    # Plot 1: Loss curves
    axes[0].plot(metrics['steps'], metrics['train_loss'], label='Train', marker='o')
    axes[0].plot(metrics['steps'], metrics['val_loss'], label='Validation', marker='s')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training & Validation Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Perplexity
    axes[1].plot(metrics['steps'], metrics['perplexity'], color='green', marker='o')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Perplexity')
    axes[1].set_title('Perplexity (lower = better)')
    axes[1].grid(True, alpha=0.3)
    
    # Plot 3: Accuracy
    axes[2].plot(metrics['steps'], metrics['accuracy'], color='orange', marker='o')
    axes[2].set_xlabel('Step')
    axes[2].set_ylabel('Accuracy (%)')
    axes[2].set_title('Next-Token Prediction Accuracy')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print final numbers
    print(f"\nFinal metrics:")
    print(f"  Train loss:  {metrics['train_loss'][-1]:.4f}")
    print(f"  Val loss:    {metrics['val_loss'][-1]:.4f}")
    print(f"  Perplexity:  {metrics['perplexity'][-1]:.2f}")
    print(f"  Accuracy:    {metrics['accuracy'][-1]:.1f}%")
except Exception as e:
    print(f"Could not plot metrics: {e}")
    print("Make sure training has completed first!")

## Step 4: Generate Armenian Text!

Now let's use our trained model to generate new Armenian text.

Try different settings:
- **Temperature**: Controls randomness
  - Low (0.3): Safe, repetitive text
  - Medium (0.8): Balanced, natural text
  - High (1.5): Creative, sometimes wild text
- **Prompt**: Starting text in Armenian

In [ ]:
# Generate text!
PROMPT = "Հայաստանի"     # <-- try different Armenian text here!
TEMPERATURE = 0.8          # <-- try 0.3, 0.8, or 1.5
LENGTH = 500               # <-- how many characters to generate

!python generate.py --prompt "{PROMPT}" --temperature {TEMPERATURE} --length {LENGTH}

In [ ]:
# Compare different temperatures side by side
print("=" * 60)
print("LOW TEMPERATURE (0.3) - Safe and repetitive")
print("=" * 60)
!python generate.py --prompt "Երևանը" --temperature 0.3 --length 200

print("\n" + "=" * 60)
print("MEDIUM TEMPERATURE (0.8) - Balanced")
print("=" * 60)
!python generate.py --prompt "Երևանը" --temperature 0.8 --length 200

print("\n" + "=" * 60)
print("HIGH TEMPERATURE (1.5) - Creative and wild")
print("=" * 60)
!python generate.py --prompt "Երևանը" --temperature 1.5 --length 200

## What's Next?

Congratulations! You've built and trained a language model from scratch!

Things to try:
- **More training**: Increase `max_iters` in the training step
- **Bigger model**: Try the `medium` preset (needs more GPU time)
- **Different data**: Add OSCAR or CC-100 data for more text
- **Level 2**: Try BPE tokenization for better quality (run `python data/prepare.py --tokenizer bpe`)
- **Experiment**: Change `n_layer`, `n_head`, `n_embd` and see what happens!

### Understanding the Results

| Metric | What it means |
|--------|---------------|
| **Loss** | How wrong the model's predictions are (lower = better) |
| **Perplexity** | How many characters the model is "confused" between (lower = better) |
| **Accuracy** | % of next characters predicted correctly (higher = better) |

A character-level model trained on Wikipedia should reach:
- Loss: 1.5 - 2.0
- Perplexity: 4 - 8
- Accuracy: 30-45%